In [1]:
#Data Preparation & Pipeline Lead

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pickle
import os

In [2]:
df = pd.read_csv('../data/Diabetes_and_LifeStyle_Dataset_.csv')

print("Dataset Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head()

Dataset Shape: (97297, 31)

Columns: ['Age', 'gender', 'ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'glucose_fasting', 'glucose_postprandial', 'insulin_level', 'hba1c', 'diabetes_risk_score', 'diabetes_stage', 'diagnosed_diabetes']


,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
1,52,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,55,50,30,93,150,2.00,5.63,23.0,No Diabetes,0
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1


In [3]:
df_clean = df.copy()

# Drop columns that would cause data leakage
df_clean = df_clean.drop(columns=['diagnosed_diabetes', 'diabetes_risk_score'])
print("Dropped leakage columns: diagnosed_diabetes, diabetes_risk_score")

# Define column types based on actual data
numeric_cols = ['Age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week',
                'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day',
                'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp',
                'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
                'triglycerides', 'glucose_fasting', 'glucose_postprandial',
                'insulin_level', 'hba1c']

categorical_cols = ['gender', 'ethnicity', 'education_level', 'income_level',
                    'employment_status', 'smoking_status', 'family_history_diabetes',
                    'hypertension_history', 'cardiovascular_history']

# Fill numeric missing values with median
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val) # Reassignment is safer
        print(f"Filled '{col}' with median: {median_val:.2f}")

# Fill categorical missing values with mode
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val) # Reassignment is safer
        print(f"Filled '{col}' with mode: {mode_val}")

constant_cols = [col for col in df_clean.columns if df_clean[col].nunique() <= 1]
if constant_cols:
    df_clean.drop(columns=constant_cols, inplace=True)
    print(f"Dropped constant columns: {constant_cols}")

print("\nMissing values remaining:", df_clean.isnull().sum().sum())
print("Shape after cleaning:", df_clean.shape)

Dropped leakage columns: diagnosed_diabetes, diabetes_risk_score

Missing values remaining: 0
Shape after cleaning: (97297, 29)


In [4]:
# 1. Initialize the new dataframe
df_encoded = df_clean.copy()

# 2. Handle "Rare" or "New" Categorical values
# This ensures that if a user enters something new in the DASH app, it maps to 'Other'
valid_ethnicities = ['White', 'Black', 'Hispanic', 'Asian'] 
df_encoded['ethnicity'] = df_encoded['ethnicity'].apply(lambda x: x if x in valid_ethnicities else 'Other')

# 3. Binary columns - Manual mapping for consistency
# 1 = True/Yes, 0 = False/No.
binary_mapping = {'Male': 1, 'Female': 0, True: 1, False: 0, 'Yes': 1, 'No': 0}
binary_cols = ['gender', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history']

for col in binary_cols:
    df_encoded[col] = df_encoded[col].map(binary_mapping).fillna(0).astype(int)

# 4. Multi-category columns - One-hot encode
multi_cols = ['ethnicity', 'education_level', 'income_level', 'employment_status', 'smoking_status']
df_encoded = pd.get_dummies(df_encoded, columns=multi_cols, drop_first=True)

# 5. Manual Target Mapping
risk_mapping = {
    'No Diabetes': 0, 
    'Pre-Diabetes': 1, 
    'Type 1': 2, 
    'Type 2': 3, 
    'Gestational': 4
}
df_encoded['diabetes_stage_encoded'] = df_encoded['diabetes_stage'].map(risk_mapping)

# Drop original text target
df_encoded = df_encoded.drop(columns=['diabetes_stage'])

print(f"Final shape after encoding: {df_encoded.shape}")

Final shape after encoding: (97297, 40)


In [5]:

# 1. Define the classes in the specific medical order we used
# This ensures the 'target_encoder.pkl' matches our manual mapping
classes = ['No Diabetes', 'Pre-Diabetes', 'Type 1', 'Type 2', 'Gestational']

# 2. Create a LabelEncoder and "force" it to use our order
target_le = LabelEncoder()
target_le.classes_ = np.array(classes)

print("Target encoding mapping (Ordinal/Medical Order):")
for i, cls in enumerate(target_le.classes_):
    print(f"  {i} → {cls}")

# 3. Save the encoder artifact
# Your DASH app will use this to turn model numbers (0, 1, 2...) back into text
os.makedirs('../artifacts', exist_ok=True)
with open('../artifacts/target_encoder.pkl', 'wb') as f:
    pickle.dump(target_le, f)

print("\nTarget encoder saved successfully to ../artifacts/target_encoder.pkl")

Target encoding mapping (Ordinal/Medical Order):
  0 → No Diabetes
  1 → Pre-Diabetes
  2 → Type 1
  3 → Type 2
  4 → Gestational

Target encoder saved successfully to ../artifacts/target_encoder.pkl


In [6]:
df_no_outliers = df_encoded.copy()

for col in numeric_cols:
    Q1 = df_no_outliers[col].quantile(0.25)
    Q3 = df_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df_no_outliers[col] < lower) | (df_no_outliers[col] > upper)).sum()
    if outliers > 0:
        df_no_outliers[col] = df_no_outliers[col].clip(lower, upper)
        print(f"'{col}': capped {outliers} outliers")

print("\nShape after outlier handling:", df_no_outliers.shape)

'alcohol_consumption_per_week': capped 443 outliers
'physical_activity_minutes_per_week': capped 3115 outliers
'diet_score': capped 325 outliers
'sleep_hours_per_day': capped 870 outliers
'screen_time_hours_per_day': capped 302 outliers
'bmi': capped 729 outliers
'waist_to_hip_ratio': capped 265 outliers
'systolic_bp': capped 517 outliers
'diastolic_bp': capped 717 outliers
'heart_rate': capped 836 outliers
'cholesterol_total': capped 296 outliers
'hdl_cholesterol': capped 555 outliers
'ldl_cholesterol': capped 340 outliers
'triglycerides': capped 291 outliers
'glucose_fasting': capped 722 outliers
'glucose_postprandial': capped 617 outliers
'insulin_level': capped 322 outliers
'hba1c': capped 599 outliers

Shape after outlier handling: (97297, 40)


In [7]:
# 1. Use df_encoded (all data) for the split
X = df_encoded.drop(columns=['diabetes_stage_encoded'])
y = df_encoded['diabetes_stage_encoded']

# 2. Perform the Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# 3. Corrected Outlier Removal (Calculate on Train ONLY)
# This prevents Data Leakage
def remove_outliers_train(df_X, df_y, columns):
    indices_to_keep = pd.Series(True, index=df_X.index)
    for col in columns:
        Q1 = df_X[col].quantile(0.25)
        Q3 = df_X[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        indices_to_keep &= (df_X[col] >= lower_bound) & (df_X[col] <= upper_bound)
    return df_X[indices_to_keep], df_y[indices_to_keep]

# Apply to training set only
# List your numeric medical columns here (BMI, Glucose, etc.)
medical_cols = ['bmi', 'hba1c', 'glucose_fasting', 'glucose_postprandial'] 
X_train, y_train = remove_outliers_train(X_train, y_train, medical_cols)

print(f"Final Train size (no outliers): {X_train.shape[0]:,} rows")
print(f"Validation size: {X_val.shape[0]:,} rows")
print(f"Test size: {X_test.shape[0]:,} rows")

Final Train size (no outliers): 66,588 rows
Validation size: 14,595 rows
Test size: 14,595 rows


In [8]:
# 1. Initialize and fit the scaler
scaler = StandardScaler()
feature_cols = X_train.columns.tolist()

# 2. Scale ONLY the numeric columns (Selective Scaling)
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

os.makedirs('../artifacts', exist_ok=True)
with open('../artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# 4. Save the feature column list (Ensures the app uses the right order)
with open('../artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("Scaling complete and artifacts saved to ../artifacts/")
print(f"Mean of scaled training features: {X_train_scaled[numeric_cols].mean().mean().round(4)}")

Scaling complete and artifacts saved to ../artifacts/
Mean of scaled training features: -0.0


In [9]:
os.makedirs('../data/processed', exist_ok=True)

# Unscaled (for tree models - Role 3)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

# Scaled (for K-Means - Role 4)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_val_scaled.to_csv('../data/processed/X_val_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)

# Target labels
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler and feature list
with open('../artifacts/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../artifacts/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("All files saved!")
for f in os.listdir('../data/processed'):
    print(f"  {f}")

All files saved!
  X_test.csv
  X_test_scaled.csv
  X_train.csv
  X_train_scaled.csv
  X_val.csv
  X_val_scaled.csv
  y_test.csv
  y_train.csv
  y_val.csv


In [10]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Original dataset shape:      {df.shape}")

# Calculate total rows remaining across all splits
total_processed = len(X_train) + len(X_val) + len(X_test)
print(f"Total rows after processing: {total_processed:,}")

print(f"Features used:               {len(feature_cols)}")
print(f"Target classes:              {list(target_le.classes_)}")

print(f"\nSplit sizes (Outliers removed from Train):")
print(f"  Train:      {len(X_train):,} rows")
print(f"  Validation: {len(X_val):,} rows")
print(f"  Test:       {len(X_test):,} rows")

print("\nArtifacts saved to ../artifacts/:")
print("  - scaler.pkl (StandardScaler)")
print("  - target_encoder.pkl (Manual LabelEncoder)")
print("  - feature_columns.pkl (List of final features)")

PREPROCESSING SUMMARY
Original dataset shape:      (97297, 31)
Total rows after processing: 95,778
Features used:               39
Target classes:              [np.str_('No Diabetes'), np.str_('Pre-Diabetes'), np.str_('Type 1'), np.str_('Type 2'), np.str_('Gestational')]

Split sizes (Outliers removed from Train):
  Train:      66,588 rows
  Validation: 14,595 rows
  Test:       14,595 rows

Artifacts saved to ../artifacts/:
  - scaler.pkl (StandardScaler)
  - target_encoder.pkl (Manual LabelEncoder)
  - feature_columns.pkl (List of final features)


In [12]:


print("Creating additional artifacts for Dash app...")

# 1. Feature means (for autofilling missing features)
feature_means = X_train.mean().to_dict()
with open('../artifacts/feature_means.pkl', 'wb') as f:
    pickle.dump(feature_means, f)
print(" Saved feature_means.pkl")

# 2. Cluster labels mapping
cluster_labels = {
    0: "Low Risk",      
    1: "High Risk",     
    2: "Moderate Risk" 
}
with open('../artifacts/cluster_labels.pkl', 'wb') as f:
    pickle.dump(cluster_labels, f)
print(" Saved cluster_labels.pkl")

# 3. Verify all artifacts exist
print("\n Artifacts directory contents:")
for f in sorted(os.listdir('../artifacts')):
    size = os.path.getsize(os.path.join('../artifacts', f))
    print(f"   - {f} ({size:,} bytes)")

print("\n All artifacts ready for deployment!")

Creating additional artifacts for Dash app...
 Saved feature_means.pkl
 Saved cluster_labels.pkl

 Artifacts directory contents:
   - cluster_labels.pkl (61 bytes)
   - feature_columns.pkl (836 bytes)
   - feature_means.pkl (1,187 bytes)
   - kmeans_model.pkl (268,135 bytes)
   - model_1.pkl (55,729 bytes)
   - model_2.pkl (76,063,937 bytes)
   - model_3.pkl (1,202,215 bytes)
   - scaler.pkl (1,336 bytes)
   - target_encoder.pkl (477 bytes)
   - xgboost_model.pkl (4,920,811 bytes)

 All artifacts ready for deployment!
